# TraceGuard AI - Simple Runner

Run the initialization cell once. Then edit only the query cell and rerun
it for each new query.

Queries are routed automatically -- a known-entity lookup, a traceability
question, a baseline check, or a full free-text impact analysis all go
through the same `orch.run(query)` call. If nothing routes confidently,
the Agent Planner attempts to compose a fallback plan from the underlying
tools before asking for clarification.

In [ ]:
import sys
from pathlib import Path

# Notebook is expected under <repo>/notebooks and module under <repo>/src.
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.traceguard_v2 import TraceGuard

traceguard = TraceGuard(data_path=repo_root / 'data')
print('TraceGuard ready.')

In [ ]:
from src.orchestrator import Orchestrator

orch = Orchestrator(traceguard)
print('Orchestrator ready.')

## Enter a new query and run only this cell

In [ ]:
query = """
Change braking system axle brake requirements and functionality.
""".strip()

result = orch.run(query)

print(f"Workflow:    {result.workflow}")
print(f"Confidence:  {result.confidence}")
print(f"Success:     {result.success}")
print(f"Steps run:   {' -> '.join(result.steps_run) if result.steps_run else '(none)'}")
if result.plan_reasoning:
    print(f"Plan reason: {result.plan_reasoning}")
print()

response = result.final_response

if isinstance(response, dict) and "impact_report_df" in response:
    # full_impact_analysis (whether reached directly or via a
    # planner-composed plan) -- show the rich report.
    display(response["impact_report_df"])
    print("\nOverall assessment:")
    print(response.get("overall_assessment"))
    if "evidence_fusion" in response:
        ef = response["evidence_fusion"]
        print("\nEvidence fusion:", ef["status"])
        print("Agreement:", ef["agreement_release_ids"])
        print("LLM-only:", ef["llm_only_release_ids"])
        print("Evidence-only:", ef["evidence_only_release_ids"])
elif isinstance(response, dict) and "baseline_determination" in response:
    # baseline_check
    print("Baseline determination:", response["baseline_determination"]["status"])
    display(response.get("affected_baselines_df"))
elif isinstance(response, dict) and "discoveries" in response:
    # traceability_trace
    print(f"Discovered {response['discovered_count']} linked artifact(s).")
    for artifact_id in response["discoveries"]:
        print(" -", artifact_id)
elif isinstance(response, dict) and "results_by_type" in response:
    # similarity_check
    for artifact_type, rows in response["results_by_type"].items():
        if rows:
            print(f"{artifact_type}: {len(rows)} candidate(s)")
else:
    # direct_lookup, or anything else -- print as-is
    print(response)